# Evaluate Saved Model on Reconstructed Test Set (Unidirectional LSTM variant)

> This is the plain-LSTM version — for the BiLSTM variant, use `hybrid_bilstm_evaluate.ipynb`. The two are not interchangeable: `lstm_classifier.pt` weights saved by one will fail to load in the other (different layer shapes).

Standalone evaluation — no training happens here. Loads the config and model artifacts
saved by the training notebook, rebuilds the exact same test set (same window size,
same GLOBAL_CUTOFF_TIME, same scalers — loaded, not refit), runs inference through all
three saved model pieces (ResNet1D → LSTM, Feeder GNN, fusion classifier), and reports
the full metric set.

**Required files from the training notebook**, expected in the working directory:
`smart_meter_labeled.csv`, `run_config.json`, `resnet1d_extractor.pt`,
`lstm_classifier.pt`, `feeder_gnn.pt`, `fusion_classifier.pt`, `seq_scaler.joblib`,
`tab_scaler.joblib`.

`smart_meter_labeled.csv` is read-only here, same as in training.


In [31]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using device:", device)

Using device: mps


## Load the exact config used during training

In [32]:
with open("lstm_models/run_config.json") as f:
    cfg = json.load(f)

TIME_COL = cfg["TIME_COL"]
ID_COL   = cfg["ID_COL"]
RAW_FEATURE_COLS = cfg["RAW_FEATURE_COLS"]
TARGET_COLS = cfg["TARGET_COLS"]
WINDOW_SIZE = cfg["WINDOW_SIZE"]
RESNET_FILTERS = cfg["RESNET_FILTERS"]
LSTM_UNITS = cfg["LSTM_UNITS"]
MIN_METERS_FOR_GNN = cfg["MIN_METERS_FOR_GNN"]
GNN_HIDDEN_DIM = cfg["GNN_HIDDEN_DIM"]
GLOBAL_CUTOFF_TIME = pd.Timestamp(cfg["GLOBAL_CUTOFF_TIME"])

BATCH_SIZE = 256
print(f"Loaded config — window={WINDOW_SIZE}, cutoff={GLOBAL_CUTOFF_TIME}, min_meters_for_gnn={MIN_METERS_FOR_GNN}")

Loaded config — window=48, cutoff=2017-11-29 03:00:00, min_meters_for_gnn=30


## Load dataset and scalers
Scalers are LOADED, never refit — refitting on new data would silently break comparability with the training run.

In [33]:
df = pd.read_csv("Dataset/clean_data_labeled.csv")
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
df = df.sort_values([ID_COL, TIME_COL]).reset_index(drop=True)
df[RAW_FEATURE_COLS] = df[RAW_FEATURE_COLS].fillna(0)

all_meters = sorted(df[ID_COL].unique())
n_meters = len(all_meters)
meter_to_idx = {m: i for i, m in enumerate(all_meters)}

seq_scaler = joblib.load("lstm_models/seq_scaler.joblib")
tab_scaler = joblib.load("lstm_models/tab_scaler.joblib")
print(f"Meters: {n_meters}")

Meters: 43


## Rebuild rolling-window tabular features (identical to training)

In [34]:
def add_rolling_features(df: pd.DataFrame, window: int = 8) -> pd.DataFrame:
    df = df.copy()
    grouped = df.groupby(ID_COL)[RAW_FEATURE_COLS]
    roll_mean = grouped.transform(lambda s: s.rolling(window, min_periods=1).mean())
    roll_std  = grouped.transform(lambda s: s.rolling(window, min_periods=1).std().fillna(0))
    roll_mean.columns = [f"{c}_roll_mean" for c in RAW_FEATURE_COLS]
    roll_std.columns  = [f"{c}_roll_std"  for c in RAW_FEATURE_COLS]
    return pd.concat([df, roll_mean, roll_std], axis=1)

df = add_rolling_features(df)
ROLLING_COLS = [c for c in df.columns if c.endswith("_roll_mean") or c.endswith("_roll_std")]
TABULAR_COLS = RAW_FEATURE_COLS + ROLLING_COLS

## Rebuild CNN-LSTM branch test windows
Same causal windowing as training. Only the TEST portion (timestamp > GLOBAL_CUTOFF_TIME)
is kept — no need to rebuild train windows for evaluation-only use.

In [35]:
def build_windows(df: pd.DataFrame):
    X_seq, X_tab, Y, meta = [], [], [], []
    for mid, g in df.groupby(ID_COL):
        g = g.sort_values(TIME_COL).reset_index(drop=True)
        raw_vals = g[RAW_FEATURE_COLS].values
        tab_vals = g[TABULAR_COLS].values
        y_vals   = g[TARGET_COLS].values
        times    = g[TIME_COL].values
        for i in range(WINDOW_SIZE - 1, len(g)):
            X_seq.append(raw_vals[i - WINDOW_SIZE + 1: i + 1])
            X_tab.append(tab_vals[i])
            Y.append(y_vals[i])
            meta.append((mid, times[i]))
    return (np.array(X_seq, dtype=np.float32),
            np.array(X_tab, dtype=np.float32),
            np.array(Y, dtype=np.float32),
            pd.DataFrame(meta, columns=[ID_COL, TIME_COL]))

X_seq, X_tab, Y, meta = build_windows(df)
meta[TIME_COL] = pd.to_datetime(meta[TIME_COL])

test_mask_cl = (meta[TIME_COL] > GLOBAL_CUTOFF_TIME).values
X_seq_test, X_tab_test, Y_test = X_seq[test_mask_cl], X_tab[test_mask_cl], Y[test_mask_cl]
meta_test = meta[test_mask_cl].reset_index(drop=True)

n_train, w, c = X_seq.shape[0], X_seq.shape[1], X_seq.shape[2]
X_seq_test_scaled = seq_scaler.transform(X_seq_test.reshape(-1, c)).reshape(X_seq_test.shape[0], w, c)
X_tab_test_scaled = tab_scaler.transform(X_tab_test)

n_features = X_seq.shape[2]
n_targets = Y.shape[1]
print(f"CNN-LSTM test windows: {len(X_seq_test)}")

CNN-LSTM test windows: 341420


## Model definitions
Must match the training notebook's architecture exactly — only the trained weights are loaded, not the class definitions.

In [36]:
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU()
        self.shortcut = None
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.BatchNorm1d(out_channels),
            )

    def forward(self, x):
        residual = x if self.shortcut is None else self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + residual)


class ResNet1DExtractor(nn.Module):
    def __init__(self, in_channels, filters):
        super().__init__()
        blocks, prev = [], in_channels
        for f in filters:
            blocks.append(ResidualBlock1D(prev, f))
            prev = f
        self.blocks = nn.Sequential(*blocks)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.blocks(x)
        return x.transpose(1, 2)


class LSTMClassifier(nn.Module):
    """Unidirectional LSTM — matches the architecture used before the BiLSTM swap.
    Use this evaluation notebook for model weights saved from that earlier training
    run; it will NOT load weights saved by the BiLSTM version (shape mismatch on fc1
    and the LSTM's own weights), and vice versa."""
    def __init__(self, embed_dim, n_tabular, lstm_units, n_targets):
        super().__init__()
        self.lstm = nn.LSTM(input_size=embed_dim, hidden_size=lstm_units, batch_first=True)
        self.fc1 = nn.Linear(lstm_units + n_tabular, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, n_targets)

    def forward(self, embed_seq, tab_features):
        _, (h_n, _) = self.lstm(embed_seq)
        lstm_out = h_n.squeeze(0)
        combined = torch.cat([lstm_out, tab_features], dim=1)
        fused_embedding = self.relu(self.fc1(combined))
        logits = self.fc2(fused_embedding)
        return logits, fused_embedding


class FeederGATLayer(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.meter_proj = nn.Linear(in_dim, hidden_dim)
        self.feeder_proj = nn.Linear(in_dim, hidden_dim)
        self.attn = nn.Linear(hidden_dim * 2, 1)
        self.update_meter = nn.Linear(hidden_dim * 2, hidden_dim)
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, meter_feats, feeder_feats, presence_mask):
        batch, n_m, _ = meter_feats.shape
        h_meter = self.meter_proj(meter_feats)
        h_feeder = self.feeder_proj(feeder_feats)
        h_feeder_exp = h_feeder.unsqueeze(1).expand(-1, n_m, -1)
        attn_scores = self.leaky_relu(self.attn(torch.cat([h_meter, h_feeder_exp], dim=-1))).squeeze(-1)
        attn_scores = attn_scores.masked_fill(presence_mask == 0, float("-inf"))
        attn_weights = torch.softmax(attn_scores, dim=1)
        feeder_update = torch.einsum("bn,bnh->bh", attn_weights, h_meter)
        feeder_update_exp = feeder_update.unsqueeze(1).expand(-1, n_m, -1)
        meter_embeddings = torch.relu(
            self.update_meter(torch.cat([h_meter, feeder_update_exp], dim=-1))
        )
        return meter_embeddings, feeder_update


class FusionClassifier(nn.Module):
    def __init__(self, in_dim, n_targets):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, n_targets)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

## Load trained weights (no training — inference only)

In [37]:
extractor = ResNet1DExtractor(n_features, RESNET_FILTERS).to(device)
extractor.load_state_dict(torch.load("lstm_models/resnet1d_extractor.pt", map_location=device))
extractor.eval()

n_tabular = X_tab_test_scaled.shape[1]
embed_dim = RESNET_FILTERS[-1]
lstm_classifier = LSTMClassifier(embed_dim, n_tabular, LSTM_UNITS, n_targets).to(device)
lstm_classifier.load_state_dict(torch.load("lstm_models/lstm_classifier.pt", map_location=device))
lstm_classifier.eval()

node_feat_dim = len(RAW_FEATURE_COLS) + 1
gnn_layer = FeederGATLayer(node_feat_dim, GNN_HIDDEN_DIM).to(device)
gnn_layer.load_state_dict(torch.load("lstm_models/feeder_gnn.pt", map_location=device))
gnn_layer.eval()

fusion_in_dim = 64 + GNN_HIDDEN_DIM  # LSTMClassifier's fc1 output width + GNN embedding width
fusion_model = FusionClassifier(fusion_in_dim, n_targets).to(device)
fusion_model.load_state_dict(torch.load("lstm_models/fusion_classifier.pt", map_location=device))
fusion_model.eval()

print("All model weights loaded — evaluation only, nothing is being trained here.")

All model weights loaded — evaluation only, nothing is being trained here.


## Run CNN-LSTM branch inference on the test set

In [38]:
def extract_cl_embeddings(X_seq_scaled, X_tab_scaled, batch_size=BATCH_SIZE):
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(X_seq_scaled), batch_size):
            seq_b = torch.tensor(X_seq_scaled[i:i+batch_size], dtype=torch.float32).to(device)
            tab_b = torch.tensor(X_tab_scaled[i:i+batch_size], dtype=torch.float32).to(device)
            emb_seq = extractor(seq_b)
            _, fused = lstm_classifier(emb_seq, tab_b)
            embeddings.append(fused.cpu().numpy())
    return np.concatenate(embeddings, axis=0)

cl_embed_test = extract_cl_embeddings(X_seq_test_scaled, X_tab_test_scaled)
print(f"CNN-LSTM test embeddings: {cl_embed_test.shape}")

CNN-LSTM test embeddings: (341420, 64)


## Rebuild GNN test graph snapshots (same severe-dropout exclusion logic)

In [39]:
meters_per_timestamp = df.groupby(TIME_COL)[ID_COL].nunique()
usable_timestamps = meters_per_timestamp[meters_per_timestamp >= MIN_METERS_FOR_GNN].index
usable_timestamps_dt = pd.to_datetime(usable_timestamps)

pivot = df.pivot_table(index=TIME_COL, columns=ID_COL, values=RAW_FEATURE_COLS)
presence = (df.assign(_present=1)
              .pivot_table(index=TIME_COL, columns=ID_COL, values="_present", fill_value=0))
pivot = pivot.sort_index().ffill().fillna(0)
presence = presence.reindex(pivot.index).fillna(0)

pivot = pivot.loc[usable_timestamps]
presence = presence.loc[usable_timestamps]

n_feat = len(RAW_FEATURE_COLS)
node_feat_dim = n_feat + 1

meter_features = np.zeros((len(usable_timestamps), n_meters, node_feat_dim), dtype=np.float32)
for m in all_meters:
    idx = meter_to_idx[m]
    meter_features[:, idx, :n_feat] = pivot[[(f, m) for f in RAW_FEATURE_COLS]].values
    meter_features[:, idx, n_feat] = presence[m].values

present_counts = presence.values.sum(axis=1, keepdims=True)
present_counts_safe = np.clip(present_counts, 1, None)
feeder_raw = (meter_features[:, :, :n_feat] * presence.values[:, :, None]).sum(axis=1) / present_counts_safe
feeder_frac_present = present_counts / n_meters
feeder_features = np.concatenate([feeder_raw, feeder_frac_present], axis=1).astype(np.float32)

gnn_test_mask_t = (usable_timestamps_dt > GLOBAL_CUTOFF_TIME)
meter_features_test = meter_features[gnn_test_mask_t]
feeder_features_test = feeder_features[gnn_test_mask_t]
presence_test = presence.values.astype(np.float32)[gnn_test_mask_t]
usable_test_ts = usable_timestamps_dt[gnn_test_mask_t]

print(f"GNN test timestamps: {gnn_test_mask_t.sum()}")

GNN test timestamps: 7960


In [40]:
def extract_gnn_embeddings(meter_feats, feeder_feats, presence_mask, batch_size=64):
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(meter_feats), batch_size):
            mf = torch.tensor(meter_feats[i:i+batch_size], dtype=torch.float32).to(device)
            ff = torch.tensor(feeder_feats[i:i+batch_size], dtype=torch.float32).to(device)
            pr = torch.tensor(presence_mask[i:i+batch_size], dtype=torch.float32).to(device)
            emb, _ = gnn_layer(mf, ff, pr)
            embeddings.append(emb.cpu().numpy())
    return np.concatenate(embeddings, axis=0)

gnn_embed_test = extract_gnn_embeddings(meter_features_test, feeder_features_test, presence_test)
print(f"GNN test embeddings: {gnn_embed_test.shape}")

GNN test embeddings: (7960, 43, 32)


## Build the fusion test set (same intersection logic as training)

In [41]:
def build_fusion_set(cl_embed, cl_meta, cl_labels, gnn_embed, usable_ts_subset, presence_subset):
    ts_to_pos = {t: i for i, t in enumerate(usable_ts_subset)}
    fused_X, fused_Y = [], []
    for i in range(len(cl_meta)):
        mid, t = cl_meta.iloc[i][ID_COL], cl_meta.iloc[i][TIME_COL]
        if t not in ts_to_pos:
            continue
        t_pos = ts_to_pos[t]
        m_idx = meter_to_idx[mid]
        if presence_subset[t_pos, m_idx] == 0:
            continue
        fused_X.append(np.concatenate([cl_embed[i], gnn_embed[t_pos, m_idx]]))
        fused_Y.append(cl_labels[i])
    return np.array(fused_X, dtype=np.float32), np.array(fused_Y, dtype=np.float32)

X_fusion_test, Y_fusion_test = build_fusion_set(
    cl_embed_test, meta_test, Y_test, gnn_embed_test, usable_test_ts, presence_test)

print(f"Fusion test samples: {X_fusion_test.shape}")

Fusion test samples: (341289, 96)


## Run final inference and compute the full metric set

In [42]:
with torch.no_grad():
    test_logits = fusion_model(torch.tensor(X_fusion_test, dtype=torch.float32).to(device))
    test_proba = torch.sigmoid(test_logits).cpu().numpy()

fusion_predictions = (test_proba >= 0.5).astype(int)

metrics_rows = []
confusion_matrices = {}  # label -> 2x2 array, kept for saving/plotting later

for i, label in enumerate(TARGET_COLS):
    y_true = Y_fusion_test[:, i]
    y_pred = fusion_predictions[:, i]
    y_proba = test_proba[:, i]

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    confusion_matrices[label] = cm

    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall    = recall_score(y_true, y_pred, zero_division=0)
    f1        = f1_score(y_true, y_pred, zero_division=0)

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    if len(np.unique(y_true)) < 2:
        roc_auc, pr_auc, mcc = np.nan, np.nan, np.nan
    else:
        roc_auc = roc_auc_score(y_true, y_proba)
        pr_auc = average_precision_score(y_true, y_proba)
        mcc = matthews_corrcoef(y_true, y_pred)

    print(f"\n--- {label} ---")
    print(cm)
    print(f"TP={tp}  TN={tn}  FP={fp}  FN={fn}")
    print(f"Accuracy: {accuracy:.4f}  Precision: {precision:.4f}  Recall: {recall:.4f}  F1: {f1:.4f}")
    print(f"Specificity: {specificity:.4f}  FPR: {fpr:.4f}  FNR: {fnr:.4f}")
    print(f"ROC-AUC: {roc_auc:.4f}  PR-AUC: {pr_auc:.4f}  MCC: {mcc:.4f}")

    metrics_rows.append({
        "Tamper Type": label, "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1": f1,
        "Specificity": specificity, "FPR": fpr, "FNR": fnr,
        "ROC-AUC": roc_auc, "PR-AUC": pr_auc, "MCC": mcc,
    })

metrics_df = pd.DataFrame(metrics_rows).set_index("Tamper Type")

count_cols = ["TP", "TN", "FP", "FN"]
rate_cols = ["Accuracy", "Precision", "Recall", "F1", "Specificity", "FPR", "FNR",
             "ROC-AUC", "PR-AUC", "MCC"]

overall_row = {}
overall_row.update(metrics_df[count_cols].sum().to_dict())
overall_row.update(metrics_df[rate_cols].mean(skipna=True).to_dict())

metrics_df.loc["OVERALL (macro avg)"] = overall_row
metrics_df


--- Current Imbalance ---
[[230133   2553]
 [  2493 106110]]
TP=106110  TN=230133  FP=2553  FN=2493
Accuracy: 0.9852  Precision: 0.9765  Recall: 0.9770  F1: 0.9768
Specificity: 0.9890  FPR: 0.0110  FNR: 0.0230
ROC-AUC: 0.9985  PR-AUC: 0.9977  MCC: 0.9659

--- Voltage Unbalance ---
[[341216     17]
 [    50      6]]
TP=6  TN=341216  FP=17  FN=50
Accuracy: 0.9998  Precision: 0.2609  Recall: 0.1071  F1: 0.1519
Specificity: 1.0000  FPR: 0.0000  FNR: 0.8929
ROC-AUC: 0.9981  PR-AUC: 0.1725  MCC: 0.1671

--- Missing Potential ---
[[341249      1]
 [    39      0]]
TP=0  TN=341249  FP=1  FN=39
Accuracy: 0.9999  Precision: 0.0000  Recall: 0.0000  F1: 0.0000
Specificity: 1.0000  FPR: 0.0000  FNR: 1.0000
ROC-AUC: 0.8711  PR-AUC: 0.0481  MCC: -0.0000

--- High Voltage ---
[[341221     23]
 [    40      5]]
TP=5  TN=341221  FP=23  FN=40
Accuracy: 0.9998  Precision: 0.1786  Recall: 0.1111  F1: 0.1370
Specificity: 0.9999  FPR: 0.0001  FNR: 0.8889
ROC-AUC: 0.9654  PR-AUC: 0.0620  MCC: 0.1408

--- Low

,TP,TN,FP,FN,Accuracy,Precision,Recall,F1,Specificity,FPR,FNR,ROC-AUC,PR-AUC,MCC
Tamper Type,,,,,,,,,,,,,,
Current Imbalance,106110,230133,2553,2493,0.985215,0.976505,0.977045,0.976775,0.989028,0.010972,0.022955,0.998498,0.997676,0.965931
Voltage Unbalance,6,341216,17,50,0.999804,0.260870,0.107143,0.151899,0.999950,0.000050,0.892857,0.998056,0.172466,0.167098
Missing Potential,0,341249,1,39,0.999883,0.000000,0.000000,0.000000,0.999997,0.000003,1.000000,0.871149,0.048148,-0.000018
High Voltage,5,341221,23,40,0.999815,0.178571,0.111111,0.136986,0.999933,0.000067,0.888889,0.965429,0.061966,0.140770
Low Voltage,283,340911,21,74,0.999722,0.930921,0.792717,0.856278,0.999938,0.000062,0.207283,0.998520,0.921424,0.858911
Over Current,2,341225,0,62,0.999818,1.000000,0.031250,0.060606,1.000000,0.000000,0.968750,0.808540,0.060152,0.176761
Very Low PF,94973,245122,603,591,0.996501,0.993691,0.993816,0.993753,0.997546,0.002454,0.006184,0.999939,0.999844,0.991324
Neutral Disturbance,74,341208,2,5,0.999979,0.973684,0.936709,0.954839,0.999994,0.000006,0.063291,0.997931,0.951452,0.955007
CT Reversal,104516,236170,359,244,0.998233,0.996577,0.997671,0.997124,0.998482,0.001518,0.002329,0.999975,0.999957,0.995849


## Positive-only breakdown (per tamper type)

A separate, additional view — restricted to only the rows where the TRUE label is 1 for
that tamper type. This does NOT replace the full-test-set metrics above; it can't,
because with no negative rows left, Precision, Specificity, FPR, ROC-AUC, PR-AUC, and MCC
are all undefined (there's nothing for a false positive to be measured against). What
this view DOES show: of the actual tamper events of this type, how many were caught
(TP) vs missed (FN) — i.e. a direct look at recall, broken into raw counts rather than a
single ratio.

In [43]:
positive_breakdown_rows = []

for i, label in enumerate(TARGET_COLS):
    y_true = Y_fusion_test[:, i]
    y_pred = fusion_predictions[:, i]

    positive_mask = (y_true == 1)
    n_positive = int(positive_mask.sum())

    if n_positive == 0:
        tp_pos, fn_pos, recall_pos = 0, 0, np.nan
    else:
        tp_pos = int((y_pred[positive_mask] == 1).sum())
        fn_pos = int((y_pred[positive_mask] == 0).sum())
        recall_pos = tp_pos / n_positive

    print(f"{label}: total positives={n_positive}  TP={tp_pos}  FN={fn_pos}  "
          f"recall={recall_pos:.4f}" if n_positive > 0 else f"{label}: no positive instances in test set")

    positive_breakdown_rows.append({
        "Tamper Type": label, "Total Positives": n_positive,
        "TP": tp_pos, "FN": fn_pos, "Recall": recall_pos,
    })

positive_breakdown_df = pd.DataFrame(positive_breakdown_rows).set_index("Tamper Type")
positive_breakdown_df

Current Imbalance: total positives=108603  TP=106110  FN=2493  recall=0.9770
Voltage Unbalance: total positives=56  TP=6  FN=50  recall=0.1071
Missing Potential: total positives=39  TP=0  FN=39  recall=0.0000
High Voltage: total positives=45  TP=5  FN=40  recall=0.1111
Low Voltage: total positives=357  TP=283  FN=74  recall=0.7927
Over Current: total positives=64  TP=2  FN=62  recall=0.0312
Very Low PF: total positives=95564  TP=94973  FN=591  recall=0.9938
Neutral Disturbance: total positives=79  TP=74  FN=5  recall=0.9367
CT Reversal: total positives=104760  TP=104516  FN=244  recall=0.9977
CT Bypass: total positives=187856  TP=187307  FN=549  recall=0.9971
CT Open: total positives=15777  TP=14455  FN=1322  recall=0.9162
Missing Value: no positive instances in test set


,Total Positives,TP,FN,Recall
Tamper Type,,,,
Current Imbalance,108603,106110,2493,0.977045
Voltage Unbalance,56,6,50,0.107143
Missing Potential,39,0,39,0.000000
High Voltage,45,5,40,0.111111
Low Voltage,357,283,74,0.792717
Over Current,64,2,62,0.031250
Very Low PF,95564,94973,591,0.993816
Neutral Disturbance,79,74,5,0.936709
CT Reversal,104760,104516,244,0.997671


## Save results
Only outputs are written here — no model weights, no changes to `smart_meter_labeled.csv`.

In [44]:
import os 
os.makedirs("lstm_evaluate", exist_ok=True)
metrics_df.to_csv("lstm_evaluate/evaluation_results.csv", index_label="Tamper Type")
positive_breakdown_df.to_csv("lstm_evaluate/positive_only_breakdown.csv", index_label="Tamper Type")

pred_out = meta_test.copy()
fusable_mask = []
usable_ts_set = set(usable_test_ts)
for i in range(len(meta_test)):
    mid, t = meta_test.iloc[i][ID_COL], meta_test.iloc[i][TIME_COL]
    fusable_mask.append(t in usable_ts_set)
fusable_mask = np.array(fusable_mask)

pred_out = meta_test[fusable_mask].reset_index(drop=True)

long_rows = []
for i in range(len(pred_out)):
    mid, t = pred_out.iloc[i][ID_COL], pred_out.iloc[i][TIME_COL]
    for j, label in enumerate(TARGET_COLS):
        long_rows.append({
            ID_COL: mid,
            TIME_COL: t,
            "Tamper Type": label,
            "true": Y_fusion_test[i, j],
            "pred": fusion_predictions[i, j],
            "proba": test_proba[i, j],
        })

pred_out_long = pd.DataFrame(long_rows)
pred_out_long.to_csv("lstm_evaluate/evaluation_predictions.csv", index=False)

print("Saved lstm_evaluate/evaluation_results.csv, lstm_evaluate/evaluation_predictions.csv")
print("No model weights were changed. smart_meter_labeled.csv was not modified.")

import matplotlib.pyplot as plt

PLOT_LABELS = [l for l in TARGET_COLS if l != "Missing Value"]
plot_label_idx = [TARGET_COLS.index(l) for l in PLOT_LABELS]

np.savez("lstm_evaluate/confusion_matrices.npz", **{label: cm for label, cm in confusion_matrices.items()})

# OVERALL = per-sample "any tamper" view, NOT a sum of the per-type matrices.
# Summing per-type matrices would total n_samples x n_tamper_types, because each sample
# yields one binary decision PER type in a multi-label setup — a valid micro-average over
# label-decisions, but its unit is "decisions", which reads as an inflated sample count
# next to the per-type panels. This framing totals exactly the test sample count and
# answers the operational question: does an inspector get sent to this meter?
from sklearn.metrics import confusion_matrix as _cm_fn

y_true_any = (Y_fusion_test[:, plot_label_idx].sum(axis=1) > 0).astype(int)
y_pred_any = (fusion_predictions[:, plot_label_idx].sum(axis=1) > 0).astype(int)
y_score_any = test_proba[:, plot_label_idx].max(axis=1)

overall_cm = _cm_fn(y_true_any, y_pred_any, labels=[0, 1])
assert overall_cm.sum() == len(Y_fusion_test), "OVERALL matrix must total the test sample count"
plot_entities = PLOT_LABELS + ["OVERALL"]
cm_for_plot = {**{l: confusion_matrices[l] for l in PLOT_LABELS}, "OVERALL": overall_cm}

n_entities = len(plot_entities)
n_cols = 4
n_rows = int(np.ceil(n_entities / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows))
axes = axes.flatten()

for i, label in enumerate(plot_entities):
    cm = cm_for_plot[label]
    ax = axes[i]
    ax.imshow(cm, cmap="Greens" if label == "OVERALL" else "Blues")
    ax.set_title("OVERALL (any tamper)" if label == "OVERALL" else label, fontsize=10)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred 0", "Pred 1"], fontsize=8)
    ax.set_yticks([0, 1]); ax.set_yticklabels(["True 0", "True 1"], fontsize=8)
    for r in range(2):
        for c in range(2):
            ax.text(c, r, str(cm[r, c]), ha="center", va="center",
                     color="white" if cm[r, c] > cm.max() / 2 else "black", fontsize=10)

for j in range(n_entities, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig("lstm_evaluate/confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.close()

print("Saved lstm_evaluate/confusion_matrices.npz (raw arrays), lstm_evaluate/confusion_matrices.png (Missing Value excluded, OVERALL added)")

from sklearn.metrics import precision_recall_curve

pr_curve_data = {}

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows))
axes = axes.flatten()

for i, label in enumerate(PLOT_LABELS):
    col = TARGET_COLS.index(label)
    y_true = Y_fusion_test[:, col]
    y_proba = test_proba[:, col]
    ax = axes[i]

    if len(np.unique(y_true)) < 2:
        ax.set_title(f"{label}\n(single class only)", fontsize=9)
        ax.axis("off")
        continue

    precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_proba)
    pr_auc_val = average_precision_score(y_true, y_proba)
    pr_curve_data[label] = {"precision": precision_vals, "recall": recall_vals, "pr_auc": pr_auc_val}

    ax.plot(recall_vals, precision_vals)
    ax.set_title(f"{label}\nPR-AUC={pr_auc_val:.3f}", fontsize=9)
    ax.set_xlabel("Recall", fontsize=8); ax.set_ylabel("Precision", fontsize=8)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)

ax = axes[len(PLOT_LABELS)]
y_true_pooled = y_true_any        # per-sample "any tamper", same framing as the matrix
y_proba_pooled = y_score_any      # max probability across tamper types
if len(np.unique(y_true_pooled)) < 2:
    ax.set_title("OVERALL\n(single class only)", fontsize=9)
    ax.axis("off")
else:
    p_o, r_o, _ = precision_recall_curve(y_true_pooled, y_proba_pooled)
    pr_auc_o = average_precision_score(y_true_pooled, y_proba_pooled)
    pr_curve_data["OVERALL"] = {"precision": p_o, "recall": r_o, "pr_auc": pr_auc_o}
    ax.plot(r_o, p_o, color="green")
    ax.set_title(f"OVERALL (any tamper)\nPR-AUC={pr_auc_o:.3f}", fontsize=9)
    ax.set_xlabel("Recall", fontsize=8); ax.set_ylabel("Precision", fontsize=8)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)

for j in range(n_entities, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig("lstm_evaluate/pr_curves.png", dpi=150, bbox_inches="tight")
plt.close()

np.savez("lstm_evaluate/pr_curve_data.npz", **{
    f"{label}_precision": d["precision"] for label, d in pr_curve_data.items()
}, **{
    f"{label}_recall": d["recall"] for label, d in pr_curve_data.items()
})

print("Saved lstm_evaluate/pr_curves.png, lstm_evaluate/pr_curve_data.npz (Missing Value excluded, OVERALL added)")

from sklearn.metrics import roc_curve

roc_curve_data = {}

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows))
axes = axes.flatten()

for i, label in enumerate(PLOT_LABELS):
    col = TARGET_COLS.index(label)
    y_true = Y_fusion_test[:, col]
    y_proba = test_proba[:, col]
    ax = axes[i]

    if len(np.unique(y_true)) < 2:
        ax.set_title(f"{label}\n(single class only)", fontsize=9)
        ax.axis("off")
        continue

    fpr_vals, tpr_vals, _ = roc_curve(y_true, y_proba)
    roc_auc_val = roc_auc_score(y_true, y_proba)
    roc_curve_data[label] = {"fpr": fpr_vals, "tpr": tpr_vals, "roc_auc": roc_auc_val}

    ax.plot(fpr_vals, tpr_vals)
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=0.8)
    ax.set_title(f"{label}\nROC-AUC={roc_auc_val:.3f}", fontsize=9)
    ax.set_xlabel("False Positive Rate", fontsize=8); ax.set_ylabel("True Positive Rate", fontsize=8)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)

ax = axes[len(PLOT_LABELS)]
if len(np.unique(y_true_pooled)) < 2:
    ax.set_title("OVERALL\n(single class only)", fontsize=9)
    ax.axis("off")
else:
    f_o, t_o, _ = roc_curve(y_true_pooled, y_proba_pooled)
    roc_auc_o = roc_auc_score(y_true_pooled, y_proba_pooled)
    roc_curve_data["OVERALL"] = {"fpr": f_o, "tpr": t_o, "roc_auc": roc_auc_o}
    ax.plot(f_o, t_o, color="green")
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=0.8)
    ax.set_title(f"OVERALL (any tamper)\nROC-AUC={roc_auc_o:.3f}", fontsize=9)
    ax.set_xlabel("False Positive Rate", fontsize=8); ax.set_ylabel("True Positive Rate", fontsize=8)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)

for j in range(n_entities, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig("lstm_evaluate/roc_curves.png", dpi=150, bbox_inches="tight")
plt.close()

np.savez("lstm_evaluate/roc_curve_data.npz", **{
    f"{label}_fpr": d["fpr"] for label, d in roc_curve_data.items()
}, **{
    f"{label}_tpr": d["tpr"] for label, d in roc_curve_data.items()
})

print("Saved lstm_evaluate/roc_curves.png, lstm_evaluate/roc_curve_data.npz (Missing Value excluded, OVERALL added)")

bar_metrics = ["Accuracy", "Precision", "Recall", "F1", "Specificity", "FPR", "FNR", "ROC-AUC", "PR-AUC", "MCC"]

bar_df = metrics_df.loc[PLOT_LABELS, bar_metrics].copy()
bar_df.loc["OVERALL"] = metrics_df.loc[PLOT_LABELS, bar_metrics].mean(skipna=True)

fig, ax = plt.subplots(figsize=(14, 6))
bar_df.plot(kind="bar", ax=ax)
ax.set_ylabel("Score")
ax.set_title("All metrics per tamper type (Missing Value excluded; OVERALL = macro avg of plotted types)")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=8)
ax.set_ylim(-1, 1)
plt.tight_layout()
plt.savefig("lstm_evaluate/all_metrics_summary.png", dpi=150, bbox_inches="tight")
plt.close()

print("Saved lstm_evaluate/all_metrics_summary.png (Missing Value excluded, OVERALL bar group added)")

Saved lstm_evaluate/evaluation_results.csv, lstm_evaluate/evaluation_predictions.csv
No model weights were changed. smart_meter_labeled.csv was not modified.
Saved lstm_evaluate/confusion_matrices.npz (raw arrays), lstm_evaluate/confusion_matrices.png (Missing Value excluded, OVERALL added)
Saved lstm_evaluate/pr_curves.png, lstm_evaluate/pr_curve_data.npz (Missing Value excluded, OVERALL added)
Saved lstm_evaluate/roc_curves.png, lstm_evaluate/roc_curve_data.npz (Missing Value excluded, OVERALL added)
Saved lstm_evaluate/all_metrics_summary.png (Missing Value excluded, OVERALL bar group added)
